<a href="https://colab.research.google.com/github/Inkognitochek/analyze-telegram-message-history/blob/main/analyze_message_history.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install telethon openai-whisper
!apt-get install ffmpeg -y



In [ ]:
import os
import csv
import whisper
import nest_asyncio
import asyncio
from datetime import datetime
from google.colab import userdata
from telethon import TelegramClient
from telethon.errors import RPCError

nest_asyncio.apply()

# ================= SAFE SETTINGS =================
API_ID = int(userdata.get('TG_API_ID')) # paste your API_ID
API_HASH = userdata.get('TG_API_HASH')  # paste your API_HASH
CHAT_NAME = '@KOLTETSKI'                # choose username you need

START_FROM_DATE = '2026-09-24 14:00:00' # enter your date
# =================================================

clean_date_str = START_FROM_DATE.strip()
offset_date = datetime.strptime(clean_date_str, '%Y-%m-%d %H:%M:%S')

print("Шаг 1: Загружаем модель Whisper (Small)...")
model = whisper.load_model("small")

client = TelegramClient('my_telegram_session', API_ID, API_HASH, timeout=90)

async def main():
    print(f"Шаг 3: Подключаемся к {CHAT_NAME} и собираем историю с {clean_date_str}...")

    file_name = 'history_koltetski.csv'
    file_exists = os.path.exists(file_name)

    with open(file_name, 'a', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)

        if not file_exists or os.stat(file_name).st_size == 0:
            writer.writerow(['Дата', 'Кто отправил (ID)', 'Текст сообщения', 'Расшифровка голосового'])

        async for message in client.iter_messages(CHAT_NAME, offset_date=offset_date):
            voice_text = ""
            await asyncio.sleep(0.4) # anti-flood tg

            if message.voice:
                for attempt in range(1, 4):
                    try:
                        voice_path = await message.download_media(file='temp_voices/')
                        if voice_path and os.path.exists(voice_path):
                            result = model.transcribe(
                                voice_path,
                                language='ru',
                                temperature=0.0,
                                no_speech_threshold=0.6
                            )
                            voice_text = result['text']
                            os.remove(voice_path)
                        break
                    except (RPCError, Exception) as e:
                        print(f"\n Telegram притормаживает на аудио от {message.date}. Попытка {attempt}/3...")
                        await asyncio.sleep(7)
                        voice_text = f"[Голосовое пропущено из-за лимитов Telegram: {e}]"

            writer.writerow([
                message.date,
                message.sender_id,
                message.text or '',
                voice_text
            ])

    print(f"\n ГОТОВО! Все данные сохранены в файл {file_name}")

async def run():
    await client.start()
    await main()

asyncio.get_event_loop().run_until_complete(run())

In [2]:
import pandas as pd

df = pd.read_csv('history_koltetski.csv')
df.head()

,Дата,Кто отправил (ID),Текст сообщения,Расшифровка голосового
0,2026-09-24 10:56:05+00:00,1054314962,Наверное,NaN
1,2026-09-24 10:56:02+00:00,1054314962,Хз,NaN
2,2026-09-24 10:13:49+00:00,1219761145,вечером пойдешь,NaN
3,2026-09-23 15:45:01+00:00,1054314962,NaN,"Ну давай, жди, может ответить попозже, сегодн..."
4,2026-09-23 15:43:52+00:00,1219761145,NaN,"Да не, не, я стоп просто хамбек, но просто он..."


In [3]:
df.columns = ['date', 'sender_id', 'text', 'voice_text']
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
df.head()

,date,sender_id,text,voice_text
0,2022-03-12 19:51:01+00:00,1219761145,В,NaN
1,2022-03-12 19:51:16+00:00,1054314962,А,NaN
2,2022-03-12 19:51:18+00:00,1219761145,NaN,NaN
3,2022-03-12 19:51:36+00:00,1219761145,NaN,NaN
4,2022-03-12 19:53:45+00:00,1054314962,NaN,NaN


In [5]:
df['is_voice'] = df['voice_text'].fillna('').str.strip().ne('')
voice_count = df['is_voice'].sum()
total_messages = len(df)
voice_share = voice_count / total_messages

print(f"Всего сообщений: {total_messages}")
print(f"Голосовых: {voice_count}")
print(f"Voice share: {voice_share:.2%}")

df.head()

Всего сообщений: 11187
Голосовых: 1471
Voice share: 13.15%


,date,sender_id,text,voice_text,is_voice
0,2022-03-12 19:51:01+00:00,1219761145,В,NaN,False
1,2022-03-12 19:51:16+00:00,1054314962,А,NaN,False
2,2022-03-12 19:51:18+00:00,1219761145,NaN,NaN,False
3,2022-03-12 19:51:36+00:00,1219761145,NaN,NaN,False
4,2022-03-12 19:53:45+00:00,1054314962,NaN,NaN,False


In [6]:
df['sender_id'] = df['sender_id'].astype(str)

In [8]:
df['prev_sender'] = df['sender_id'].shift(1)
df['prev_time'] = df['date'].shift(1)

df['response_minutes'] = (df['date'] - df['prev_time']).dt.total_seconds() / 60
df['is_response'] = (df['sender_id'] != df['prev_sender']) & df['prev_sender'].notna()

text_response = df.loc[(df['is_response']) & (~df['is_voice']), 'response_minutes']
voice_response = df.loc[(df['is_response']) & (df['is_voice']), 'response_minutes']

print(f'Всего ответов: {df["is_response"].sum()}')
print(f'Среднее время ответа после текста: {text_response.mean():.2f} мин')
print(f'Среднее время ответа после голоса: {voice_response.mean():.2f} мин')
print(f'Медиана после текста: {text_response.median():.2f} мин')
print(f'Медиана после голоса: {voice_response.median():.2f} мин')

df[['date', 'sender_id', 'prev_sender', 'response_minutes', 'is_voice', 'is_response']].head(10)

Всего ответов: 5657
Среднее время ответа после текста: 223.40 мин
Среднее время ответа после голоса: 81.43 мин
Медиана после текста: 0.15 мин
Медиана после голоса: 0.38 мин


,date,sender_id,prev_sender,response_minutes,is_voice,is_response
0,2022-03-12 19:51:01+00:00,1219761145,NaN,NaN,False,False
1,2022-03-12 19:51:16+00:00,1054314962,1219761145,0.250000,False,True
2,2022-03-12 19:51:18+00:00,1219761145,1054314962,0.033333,False,True
3,2022-03-12 19:51:36+00:00,1219761145,1219761145,0.300000,False,False
4,2022-03-12 19:53:45+00:00,1054314962,1219761145,2.150000,False,True
5,2022-03-12 19:53:45+00:00,1054314962,1054314962,0.000000,False,False
6,2022-03-12 19:57:15+00:00,1054314962,1054314962,3.500000,False,False
7,2022-03-12 19:57:15+00:00,1054314962,1054314962,0.000000,False,False
8,2022-03-12 19:58:19+00:00,1054314962,1054314962,1.066667,False,False
9,2022-03-12 19:58:19+00:00,1054314962,1054314962,0.000000,False,False
